### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)

    pdf_files=list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents=process_all_pdfs("../data")


Found 3 PDF files to process

Processing: AI_Learning_Roadmap_Java_Developer.pdf
Loaded 10 pages

Processing: Java_SpringBoot_Microservices_Interview_Answers.pdf
Loaded 11 pages

Processing: Week1_Full_Notes_AI_Foundations.pdf
Loaded 15 pages

Total documents loaded: 36


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-10-06T19:08:42+00:00', 'author': 'Un-named', 'source': '..\\data\\pdf\\AI_Learning_Roadmap_Java_Developer.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'AI_Learning_Roadmap_Java_Developer.pdf', 'file_type': 'pdf'}, page_content="AI Learning Roadmap\nfor a Java Backend Developer\n8-week day-by-day plan with projects\nOverview\nGoal: become an applied AI engineer, someone who builds AI-powered backend systems (LLMs, \nRAG, agents). You don't need heavy math or PhD-level ML for this, and your Java and Spring skills are \na big advantage.\n• Time needed: 1.5-2 hours per day.\n• Stack: Java 21, Spring Boot, Spring AI (or LangChain4j), PostgreSQL + pgvector, Ollama (free \nlocal models), Docker.\n• How to use this document: Tick the box in the Done column after each day. Highlighted \nrows are projects.\nAI Learning Roadmap  |  Page 1"),
 Document(metadata={'producer': 'Libr

In [6]:
### Text splitting into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)
chunks

Split 36 documents into 83 chunks

Example chunk:
Content: AI Learning Roadmap
for a Java Backend Developer
8-week day-by-day plan with projects
Overview
Goal: become an applied AI engineer, someone who builds AI-powered backend systems (LLMs, 
RAG, agents). ...
Metadata: {'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-10-06T19:08:42+00:00', 'author': 'Un-named', 'source': '..\\data\\pdf\\AI_Learning_Roadmap_Java_Developer.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'AI_Learning_Roadmap_Java_Developer.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-10-06T19:08:42+00:00', 'author': 'Un-named', 'source': '..\\data\\pdf\\AI_Learning_Roadmap_Java_Developer.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'AI_Learning_Roadmap_Java_Developer.pdf', 'file_type': 'pdf'}, page_content="AI Learning Roadmap\nfor a Java Backend Developer\n8-week day-by-day plan with projects\nOverview\nGoal: become an applied AI engineer, someone who builds AI-powered backend systems (LLMs, \nRAG, agents). You don't need heavy math or PhD-level ML for this, and your Java and Spring skills are \na big advantage.\n• Time needed: 1.5-2 hours per day.\n• Stack: Java 21, Spring Boot, Spring AI (or LangChain4j), PostgreSQL + pgvector, Ollama (free \nlocal models), Docker.\n• How to use this document: Tick the box in the Done column after each day. Highlighted \nrows are projects.\nAI Learning Roadmap  |  Page 1"),
 Document(metadata={'producer': 'Libr

### Embedding and VectorstoreDB

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Abhi\Desktop\Abhishek\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """
        Intialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentance embeddings
        """
        self.model_name=model_name
        self.model=None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model{self.model_name}:{e}")
            raise
    def generate_embeddings(self,texts:List[str])->np.ndarry:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed
        
        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings
    
    def get_embedding_dimension(self)->int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_embedding_dimension

In [10]:
### Initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model:all-MiniLM-L6-v2


c:\Users\Abhi\Desktop\Abhishek\YTRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abhi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Abhi\Desktop\Abhishek\YTRAG\.venv\Lib\site-packages\huggingface_hub\file_downlo

Model loaded sucessfully. Embedding dimension: 384


### VectorStore

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self,collection_name: str="pdf_documents",persist_directory:str="../data/vectorstore"):
        """
        Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            #create persistant ChromaDB client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection:{self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) !=len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store..")

        #prepare data for chromaDB

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            # Generate unique ID
            doc_id=f"doc+{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to Collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Sucessfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        

In [12]:
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection:pdf_documents
Existing documents in collection:83


In [13]:
chunks

[Document(metadata={'producer': 'LibreOffice 24.2', 'creator': 'Writer', 'creationdate': '2026-10-06T19:08:42+00:00', 'author': 'Un-named', 'source': '..\\data\\pdf\\AI_Learning_Roadmap_Java_Developer.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'source_file': 'AI_Learning_Roadmap_Java_Developer.pdf', 'file_type': 'pdf'}, page_content="AI Learning Roadmap\nfor a Java Backend Developer\n8-week day-by-day plan with projects\nOverview\nGoal: become an applied AI engineer, someone who builds AI-powered backend systems (LLMs, \nRAG, agents). You don't need heavy math or PhD-level ML for this, and your Java and Spring skills are \na big advantage.\n• Time needed: 1.5-2 hours per day.\n• Stack: Java 21, Spring Boot, Spring AI (or LangChain4j), PostgreSQL + pgvector, Ollama (free \nlocal models), Docker.\n• How to use this document: Tick the box in the Done column after each day. Highlighted \nrows are projects.\nAI Learning Roadmap  |  Page 1"),
 Document(metadata={'producer': 'Libr

In [14]:
# Convert text to embeddings

texts=[doc.page_content for doc in chunks]

# Generate embeddings
embeddings=embedding_manager.generate_embeddings(texts)

# store in the vector store
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 83 texts...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.20s/it]


Generated embeddings with shape:(83, 384)
Adding 83 documents to vector store..
Sucessfully added 83 documents to vector store
Total documents in collection: 166


### Reteriver pipeline from vectorstore

In [28]:
class RAGRetriver:
    """Handles query-based retrieval from the vector store"""
    def __init__(self,vector_store: VectorStore, embedding_manager:EmbeddingManager):
        """
        Initialize the retriver
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager

    def retrieve(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant documents for the query
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query:'{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate Query Embedding
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            # process results
            retrieved_docs=[]
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score=1-distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval {e}")
            return []

In [29]:
rag_retriever=RAGRetriver(vectorstore,embedding_manager)
rag_retriever

In [31]:
rag_retriever.retrieve("java spring boot interview questions")

Retrieving documents for query:'java spring boot interview questions'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.54it/s]

Generated embeddings with shape:(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc+8988c8c5_49',
  'content': 'Java Backend Interview Preparation\nPage 11\nFinal-Day Revision Checklist\n\x7f Core Java: OOP, equals/hashCode, collections, exceptions, generics, String immutability, threads, locks, volatile, atomics,\nHashMap and ConcurrentHashMap.\n\x7f Java 8: Lambda, functional interfaces, Streams, Optional, method references, CompletableFuture and default/static interface\nmethods.\n\x7f Java 17: LTS benefits, records, sealed classes, pattern matching, switch expressions and JVM/runtime improvements.\n\x7f Spring Boot: IoC/DI, beans, component scanning, auto-configuration, validation, exception handling, transactions and Spring\nSecurity/JWT.\n\x7f Microservices: discovery, gateway, REST, Kafka, circuit breaker, retry, timeout, Saga, idempotency and observability.\n\x7f Kafka: partitions, consumer groups, offsets, rebalancing, DLT, retries, delivery semantics and DB-failure handling.\n\x7f SQL/JPA: joins, GROUP BY/HAVING, indexes, execution plans, second

### Inetgration VectorDB Context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
llm=ChatGroq(model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

# Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # generate the answer using GROQ LLM
    prompt=""" Use the following context to answer the question concisely.
        Context:
        {context} 
        Question:{query}       
        Answer:"""
